[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs_ar/Lab_1_Elimination_of_Constants_AR.ipynb)


In [ ]:
# 🚀 تهيئة بيئة Google Colab
# إذا كنت تشغل هذا النوت بوك على Google Colab، شغل هذه الخلية لتحميل ملفات المستودع والاختبارات تلقائياً:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] تم تجهيز بيئة Google Colab بنجاح والاتصال بمستودع المشروع!')
except NameError:
    pass


<div dir="rtl" style="text-align:right">

# 📡 اللاب 1: تكوين المعادلات التفاضلية الجزئية بحذف الثوابت الاختيارية
**المقرر:** المعادلات التفاضلية الجزئية وتطبيقاتها في ملاحة المركبات الفضائية  
**المرجع في المذكرة:** الباب الأول، القسم 1.1 و 1.2 (صفحات المذكرة 1–10 / صفحات ملف PDF: 2–11)

> 🗺️ **أين يقع هذا اللاب في خط عمل المشروع؟**  
> `خريطة الارتفاعات h(x,y)` $\longrightarrow$ **`[اللاب 1: حساب الميول p, q وكشف العوائق slope_map]`** $\longrightarrow$ `نطاق الأمان` $\longrightarrow$ `معادلة لابلاس` $\longrightarrow$ `مسار المركبة`  
> **الملف الحقيقي في المشروع:** [`rover_path_finder/terrain.py`](../rover_path_finder/terrain.py) — الدالة `slope_map(h, cell)`

---
### 🧭 قاموس الربط: الرياضيات $\longleftrightarrow$ المعنى الفيزيائي $\longleftrightarrow$ كود المشروع
| 1. الرمز في مذكرة المحاضرة | 2. المعنى الفيزيائي على أرض المريخ | 3. الكود الفعلي في `terrain.py` |
| :--- | :--- | :--- |
| $z = h(x, y)$ | ارتفاع سطح الأرض (بالمتر) عند الإحداثيات $(x, y)$ | مصفوفة الارتفاعات `h` ثنائية الأبعاد |
| $p = \frac{\partial z}{\partial x}, \quad q = \frac{\partial z}{\partial y}$ | ميل السطح باتجاه الشرق ($x$) وباتجاه الشمال ($y$) | `gy, gx = np.gradient(h, cell)` |
| $\sqrt{p^2 + q^2}$ | مقدار الانحدار الكلي للسطح (ظل زاوية الميل $\tan\theta$) | `slope = np.degrees(np.arctan(np.hypot(gx, gy)))` |
| $F(x, y, z, a, b) = 0$ | عائلة أسطح (مثلاً حفر بمراكز مختلفة $a, b$) | `Z = (X - a)**2 + (Y - b)**2` |
| $f(x, y, z, p, q) = 0$ | **المعادلة التفاضلية الجزئية (PDE):** قانون يربط الميل بالارتفاع بغض النظر عن قيمة $(a,b)$ | `residual = (P**2 + Q**2) - 4*Z` |

---
## 1. الفكرة ببساطة: لماذا نحذف الثوابت الاختيارية؟
تخيل أن على سطح المريخ مئات الحفر الدائرية. كل حفرة لها مركز مختلف $(a, b)$ على الخريطة:
$$z = (x - a)^2 + (y - b)^2$$
عندما يفحص حاسوب المركبة خريطة الارتفاعات ليقرر: **«هل هذه النقطة شديدة الانحدار (أكبر من $20^\circ$)؟»**، فإنه لا يعرف أين يقع مركز كل حفرة $(a, b)$ مسبقاً.

**حذف الثوابت الاختيارية (Elimination of Arbitrary Constants)** يحوّل معادلة تعتمد على الموقع الخاص $(a, b)$ إلى **علاقة عامة تربط الميل $(p, q)$ بالعمق $z$ مباشرةً**:
$$p^2 + q^2 = 4z$$
هذه المعادلة تخبرنا بحقيقة هندسية رائعة: **مقدار ميل جدار الحفرة يعتمد فقط على عمق النقطة $z$، ولا يعتمد إطلاقاً على مكان الحفرة $(a, b)$!** ولهذا السبب تعمل دالة `slope_map` في مشروعنا بكفاءة على أي تضاريس دون معرفة معادلاتها أو مراكز الحفر فيها.

</div>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab1

%matplotlib inline
print("تم تهيئة بيئة اللاب 1 بنجاح!")


<div dir="rtl" style="text-align:right">

---
## 2. أساسيات القسم (1.1) وقاعدة عمل القسم (1.2) من المذكرة (PDF ص 2–11)

قبل أن نحذف الثوابت، لنتقن المفاهيم الأساسية في **القسم (1.1) (صفحات المذكرة 1–5 / PDF ص 2–6)** لأنها تأتي دائماً في أسئلة الاختيارات والامتحانات:

### أولاً: رموز مونج الخمسة (Monge's Notations — صفحة 3 بالمذكرة / PDF ص 4)
في كل صفحات المذكرة، عندما تكون $z(x, y)$ دالة في متغيرين مستقلين $x$ و $y$، نستخدم الرموز القياسية التالية:
$$p = \frac{\partial z}{\partial x}, \qquad q = \frac{\partial z}{\partial y}, \qquad r = \frac{\partial^2 z}{\partial x^2}, \qquad s = \frac{\partial^2 z}{\partial x \partial y}, \qquad t = \frac{\partial^2 z}{\partial y^2}$$
*(ملاحظة جميلة للربط بالمشروع: في اللاب 4 واللاب 6 سنرى أن **معادلة لابلاس** للملاحة هي ببساطة $r + t = 0$!)*

---

### ثانياً: الفرق بين الرتبة (Order) والدرجة (Degree) — والفخ الشهير! (ص 2 بالمذكرة / PDF ص 3)
1. **الرتبة (Order):** هي رتبة **أعلى مشتقة جزئية** موجودة في المعادلة.
2. **الدرجة (Degree):** هي **أُس (قوة) أعلى مشتقة جزئية**، ولكن **بشرط جوهري:** يجب أولاً التخلص من أي جذور أو أسس كسرية أو كسور تحتوي على المشتقات!
   * **🔍 مثال الفخ في المذكرة (مثال 5 ص 1–2 / PDF ص 2–3):**
     $$\frac{\partial^2 z}{\partial x^2} = \left(1 + \frac{\partial z}{\partial y}\right)^{\frac{1}{2}}$$
     * لو تسرعت وقرأت الأس قبل التبسيط لقلت الدرجة الأولى! ولكن لوجود الأس الكسري $\frac{1}{2}$ (جذر تربيعي) على المشتقة، **يجب تربيع الطرفين أولاً**:
       $$\left(\frac{\partial^2 z}{\partial x^2}\right)^2 = 1 + \frac{\partial z}{\partial y}$$
     * الآن نقرأ أعلى مشتقة وهي $\frac{\partial^2 z}{\partial x^2}$: **الرتبة = 2 (الثانية)**، ولأنها مرفوعة للأس 2 بعد التربيع فإن **الدرجة = 2 (الثانية)**!

---

### ثالثاً: التصنيف الرباعي لمعادلات الرتبة الأولى (القسم 1.1.7 — ص 4–5 بالمذكرة / PDF ص 5–6)
كيف تفرق بين الأنواع الأربعة لمعادلات الرتبة الأولى $f(x, y, z, p, q) = 0$ بمجرد النظر؟

| النوع في المذكرة (PDF ص 5–6) | الصورة العامة | الشرط المميز في خطوة واحدة | أمثلة المذكرة (PDF ص 5–6) |
| :--- | :--- | :--- | :--- |
| **1. خطية تماماً (Linear)** | $P(x,y)p + Q(x,y)q = R(x,y)z + S(x,y)$ | $p, q$ **و كذلك $z$** كلها من الدرجة الأولى، ولا يوجد ضرب بينها ($zp$ ممنوع!) | $yx^2 p + xy^2 q = xyz + x^2 y^3$ |
| **2. نصف خطية (Semi-linear)** | $P(x,y)p + Q(x,y)q = R(x,y,z)$ | $p, q$ من الدرجة الأولى، ومعاملاتهما $P(x,y), Q(x,y)$ **تعتمد على $x, y$ فقط** (خالية من $z$)، بينما الطرف الأيمن $R$ قد يكون غير خطي في $z$ | $xyp + x^2 y q = x^2 y^2 z^2$ |
| **3. شبه خطية (Quasi-linear)** | $P(x,y,z)p + Q(x,y,z)q = R(x,y,z)$ | $p, q$ من الدرجة الأولى، ولكن معاملاتهما $P, Q$ **تحتوي على $z$** (هذه المعادلة وما سبقها تُحل جميعاً **بطريقة لاجرانج** في اللاب 2!) | $x^2 z p + y^2 z q = xy$ |
| **4. غير خطية (Non-linear)** | $f(x,y,z,p,q) = 0$ | تحتوي على تربيع $p^2, q^2$ أو حاصل ضرب $p \cdot q$ (وتُحل **بطريقة شاربت والأشكال الأربعة** في اللاب 3!) | $p^2 + q^2 = 1$ أو $pq = z$ |

---

### رابعاً: الحالات الثلاث لحذف الثوابت الاختيارية (القسم 1.2 — ص 6–9 بالمذكرة / PDF ص 7–10)
عند حذف ثوابت اختيارية من علاقة $F(x, y, z, a, b) = 0$، نشتق جزئياً بالنسبة لـ $x$ وبالنسبة لـ $y$، وتنشأ إحدى **الحالات الثلاث التالية**:

1. **الحالة (1) — عدد الثوابت $<$ عدد المتغيرات المستقلة (PDF ص 7–8):**
   * **النتيجة:** نحصل على **أكثر من معادلة تفاضلية جزئية** من الرتبة الأولى.
   * **مثال المذكرة (PDF ص 7):** $z = ax + y$ (ثابت واحد $a$ ومتغيران $x, y$):
     * نشتق بالنسبة لـ $x$: $\frac{\partial z}{\partial x} = a \implies$ نعوض عن $a$ في الأصلية فنحصل على المعادلة الأولى: $z = x\frac{\partial z}{\partial x} + y$.
     * نشتق بالنسبة لـ $y$: $\frac{\partial z}{\partial y} = 1$ (وهي خالية من $a$ أصلاً، فتمثل معادلة ثانية!).

2. **الحالة (2) — عدد الثوابت $=$ عدد المتغيرات المستقلة (PDF ص 8):**
   * **النتيجة:** نحصل على **معادلة تفاضلية جزئية واحدة وحيدة** من الرتبة الأولى.
   * **مثال المذكرة (PDF ص 8):** احذف $a, b$ من $az + b = a^2 x + y$:
     * نشتق بالنسبة لـ $x$: $a\frac{\partial z}{\partial x} = a^2 \implies \frac{\partial z}{\partial x} = a$.
     * نشتق بالنسبة لـ $y$: $a\frac{\partial z}{\partial y} = 1$.
     * نعوض عن $a = \frac{\partial z}{\partial x}$ في المعادلة الثانية فنحصل فوراً على: $\left(\frac{\partial z}{\partial x}\right)\left(\frac{\partial z}{\partial y}\right) = 1$ (أي $pq = 1$).

3. **الحالة (3) — عدد الثوابت $>$ عدد المتغيرات المستقلة (PDF ص 8–9):**
   * **النتيجة:** المشتقات الأولى وحدها لا تكفي لحذف جميع الثوابت، فنشتق مرة ثانية للحصول على معادلات من **الرتبة الثانية أو أعلى**.
   * **مثال المذكرة (PDF ص 9):** احذف الثوابت الثلاثة $a, b, c$ من $z = ax + by + cxy$:
     * المشتقات الأولى: $\frac{\partial z}{\partial x} = a + cy \;\cdots(2)$، و $\frac{\partial z}{\partial y} = b + cx \;\cdots(3)$.
     * المشتقات الثانية: $\frac{\partial^2 z}{\partial x^2} = 0$، و $\frac{\partial^2 z}{\partial y^2} = 0 \;\cdots(4)$، والمشتقة المختلطة: $\frac{\partial^2 z}{\partial x \partial y} = c \;\cdots(5)$.
     * بضرب $(2)$ في $x$ وضرب $(3)$ في $y$ وجمعهما:
       $$x\frac{\partial z}{\partial x} + y\frac{\partial z}{\partial y} = \underbrace{ax + by + cxy}_{=\,z} + cxy = z + xy\frac{\partial^2 z}{\partial x \partial y} \;\cdots(6)$$

> ⚠️ **تنبيه خطأ مطبعي في مذكرة الكلية (PDF صفحة 9 — المعادلتان 5 و 6):**
> في ملف الـ PDF (صفحة 9، السطر 6 والسطر 11)، كُتبت المشتقة المختلطة بالخطأ المطبعي $\frac{\partial^2 y}{\partial x \partial y}$ (بحرف $y$ في البسط)، والصواب رياضياً هو $\frac{\partial^2 z}{\partial x \partial y} = c$ لأن $z$ هو المتغير التابع بينما $x, y$ متغيران مستقلان!

---

### 📖 مثال محلول بالتفصيل من المذكرة (Example 1 — صفحة 8 بالمذكرة / PDF ص 9–10):
**المسألة:** كوّن المعادلة التفاضلية الجزئية بحذف الثابتين الاختياريين $a$ و $b$ من عائلة المستويات:
$$z = a x + b y + a^2 + b^2 \qquad \cdots (1)$$

**الحل خطوة بخطوة (مع سبب كل خطوة):**
* **الخطوة 1 (الاشتقاق بالنسبة لـ $x$):** نعتبر $y$ ثابتاً ونشتق المعادلة $(1)$ بالنسبة لـ $x$:
  $$p = \frac{\partial z}{\partial x} = a \qquad \cdots (2)$$
  *(لماذا؟ لأن $by + a^2 + b^2$ كلها ثوابت بالنسبة لـ $x$ فمشتقتها صفر!)*
* **الخطوة 2 (الاشتقاق بالنسبة لـ $y$):** نعتبر $x$ ثابتاً ونشتق المعادلة $(1)$ بالنسبة لـ $y$:
  $$q = \frac{\partial z}{\partial y} = b \qquad \cdots (3)$$
* **الخطوة 3 (التعويض والحذف):** نعوض عن $a = p$ وعن $b = q$ في المعادلة الأصلية $(1)$:
  $$z = x p + y q + p^2 + q^2$$
* **التحقق السريع:** هل بقيت أي رموز $a$ أو $b$؟ لا! إذن حصلنا على معادلة تفاضلية جزئية غير خطية من الرتبة الأولى والدرجة الثانية (وهي على صيغة كليروت التي سنحلها في اللاب 3!).

</div>


<div dir="rtl" style="text-align:right">

---
## 3. ✍️ تحدي الورقة والقلم 1 (Example 2 — صفحة 9 بالمذكرة / PDF ص 10)

**المسألة:** احذف الثابتين الاختياريين $a$ و $b$ من معادلة عائلة الأسطح المكافئة الدائرية (حفر المريخ):
$$z = (x - a)^2 + (y - b)^2 \qquad \cdots (1)$$

**جرّب الحل بنفسك على الورقة أولاً!** إذا احتجت مساعدة، افتح التلميحات التدريجية التالية:

<details>
<summary>💡 <b>تلميح 1: كيف نبدأ؟</b> (اضغط للفتح)</summary>

اشتق المعادلة $(1)$ جزئياً بالنسبة لـ $x$ لتحصل على $p$، ثم بالنسبة لـ $y$ لتحصل على $q$. تذكر قاعدة السلسلة: مشتقة $(x-a)^2$ بالنسبة لـ $x$ هي $2(x-a)$.
</details>

<details>
<summary>💡 <b>تلميح 2: كيف نتخلص من $(x-a)$ و $(y-b)$؟</b> (اضغط للفتح)</summary>

أصبح لديك:
$$p = 2(x - a) \quad \text{و} \quad q = 2(y - b)$$
لاحظ أن المعادلة الأصلية $(1)$ تحتوي على $(x-a)^2 + (y-b)^2$. ماذا يحدث لو ربّعت $p$ وربّعت $q$ وجمعتهما؟
</details>

<details>
<summary>💡 <b>تلميح 3: الخطوة الأخيرة</b> (اضغط للفتح)</summary>

$$p^2 + q^2 = 4(x - a)^2 + 4(y - b)^2 = 4\underbrace{\left[(x - a)^2 + (y - b)^2\right]}_{=\,z}$$
استبدل القوس الكبير بـ $z$ من المعادلة $(1)$.
</details>

اكتب المعادلة التفاضلية الجزئية الناتجة كنص في الخلية التالية (بصيغة مثل `"p^2 + q^2 = 4*z"`):

</div>


In [ ]:
# خلية تقييم: تحدي الورقة 1
### START CODE HERE ###
student_pde_1 = "p^2 + q^2 = 4*z"
### END CODE HERE ###

# يفحص المصحح الآلي معادلتك بالتعويض العددي في عشرات النقاط العشوائية:
test_lab1.check_paper_challenge_1(student_pde_1)


<div dir="rtl" style="text-align:right">

---
## 4. 🚀 تحدي الربط بالمشروع: فحص حوض الهبوط الإهليلجي

في تضاريس المريخ، قد تكون الحفرة ممتدة (إهليلجية) بمعامل انحناء $a$ شرقاً و $b$ شمالاً:
$$z = \frac{1}{2} a x^2 + \frac{1}{2} b y^2 \qquad \cdots (1)$$

**المطلوب على ورقتك:**
احذف الثابتين الاختياريين $a$ و $b$ لتكوين المعادلة التفاضلية الجزئية التي تربط الارتفاع $z$ بالميول $(p, q)$ عند أي نقطة $(x, y)$.

<details>
<summary>💡 <b>تلميح 1:</b> احسب $p$ و $q$</summary>

$$p = \frac{\partial z}{\partial x} = a x \implies a = \frac{p}{x}$$
$$q = \frac{\partial z}{\partial y} = b y \implies b = \frac{q}{y}$$
</details>

<details>
<summary>💡 <b>تلميح 2:</b> عوّض في المعادلة الأصلية</summary>

عوّض عن $a = \frac{p}{x}$ وعن $b = \frac{q}{y}$ في المعادلة $(1)$ وبسّط الكسور (يمكنك ضرب الطرفين في $2$ للتخلص من الكسور).
</details>

</div>


In [ ]:
# خلية تقييم: تحدي حوض المريخ
### START CODE HERE ###
rover_basin_pde = "x*p + y*q = 2*z"
### END CODE HERE ###

test_lab1.check_paper_challenge_2(rover_basin_pde)


<div dir="rtl" style="text-align:right">

---
## 5. 💻 من الورقة إلى الكود: التحقق العددي على شبكة التضاريس

**كيف نتحقق برمجياً من أن حلنا الورقي صحيح؟**  
إذا كانت المعادلة $p^2 + q^2 = 4z$ صحيحة، فإن الفرق بين الطرف الأيسر والطرف الأيمن (ويسمى **المتبقي Residual**):
$$\text{residual} = (p^2 + q^2) - 4z$$
يجب أن يساوي **صفراً** عند كل نقطة من نقاط الشبكة، أياً كان مركز الحفرة $(a, b)$!

أكمل الدالة `verify_crater_pde` أدناه لحساب مصفوفة المتبقي:

</div>


In [ ]:
# دالة تقييم برمجية: verify_crater_pde
def verify_crater_pde(Z: np.ndarray, P: np.ndarray, Q: np.ndarray) -> np.ndarray:
    """تحسب المتبقي (p^2 + q^2) - 4*z للتحقق من معادلة الحفرة على كامل الشبكة."""
    ### START CODE HERE ###
    residual = (P**2 + Q**2) - 4.0 * Z
    ### END CODE HERE ###
    return residual

test_lab1.test_eliminate_constants_fn(verify_crater_pde)


<div dir="rtl" style="text-align:right">

---
## 6. 🛰️ الرؤية البصرية والربط الفعلي بكود المشروع (`terrain.slope_map`)

شغّل الخلية التالية لترى شيئين معاً:
1. **على اليسار والوسط:** سطح الحفرة $z=(x-a)^2+(y-b)^2$ ومتجهات الميل $(p, q)$ التي حسبتها على الورق.
2. **على اليمين (من محرك المشروع الفعلي `rover_path_finder/terrain.py`):** كيف تستخدم دالة `slope_map` نفس المشتقتين $(p, q)$ لحساب زاوية الميل بالدرجات $\theta = \arctan\sqrt{p^2+q^2}$ وتحديد المناطق الخطرة (الحمراء $> 20^\circ$) على تضاريس المريخ!

</div>


In [ ]:
from rover_path_finder.terrain import make_terrain, slope_map

x = np.linspace(-3, 3, 40)
y = np.linspace(-3, 3, 40)
X, Y = np.meshgrid(x, y)
Z = (X - 0.5)**2 + (Y + 0.5)**2  # حفرة مركزها (a=0.5, b=-0.5)

dx = x[1] - x[0]
dy = y[1] - y[0]
# انتبه: المحور 0 هو y (الصفوف) والمحور 1 هو x (الأعمدة)
Q, P = np.gradient(Z, dy, dx)

# استدعاء دالة المشروع الحقيقية لحساب خريطة الميول على تضاريس المريخ
mars_demo = make_terrain(N=100, seed=3)
mars_slopes_deg = slope_map(mars_demo["h"], cell=1.0)

fig = plt.figure(figsize=(16, 4.8))

ax1 = fig.add_subplot(1, 3, 1, projection='3d')
ax1.plot_surface(X, Y, Z, cmap='copper', alpha=0.85)
ax1.set_title("1. Crater Surface z = (x-a)^2 + (y-b)^2", fontsize=11)
ax1.set_xlabel("x")
ax1.set_ylabel("y")

ax2 = fig.add_subplot(1, 3, 2)
ax2.contour(X, Y, Z, levels=15, cmap='copper')
ax2.quiver(X[::3, ::3], Y[::3, ::3], P[::3, ::3], Q[::3, ::3], color='crimson')
ax2.set_title(r"2. Slope Vectors (p, q) = ($\partial z/\partial x$, $\partial z/\partial y$)", fontsize=11)
ax2.set_xlabel("x")
ax2.set_ylabel("y")

ax3 = fig.add_subplot(1, 3, 3)
im = ax3.imshow(mars_slopes_deg, cmap='inferno', origin='lower')
ax3.contour(mars_demo["raw"], levels=[0.5], colors='cyan', linewidths=1.2)
plt.colorbar(im, ax=ax3, label="Slope (degrees)")
ax3.set_title("3. Project Engine: terrain.slope_map() & Hazards (cyan)", fontsize=11)
ax3.set_xlabel("Grid X")
ax3.set_ylabel("Grid Y")

plt.tight_layout()
plt.show()

#📌 ملخص اللاب 1 في 3 نقاط:
print("📌 ملخص اللاب 1:")
print("  1) p = dz/dx و q = dz/dy هما مركبتا ميل التضاريس في اتجاهي x و y.")
print("  2) حذف الثوابت الاختيارية (a, b) يعطينا قانوناً عاماً للميل (p^2 + q^2 = 4z) لا يتوقف على موقع الحفرة.")
print("  3) دالة slope_map() في المشروع تطبق هذا القانون لتحويل خريطة الارتفاعات إلى قناع عوائق (Hazard Mask).")
